## Implementation of Query Enhancement - RAG-Fusion

### Libraries, ChatOllama, Chroma vectorstore, LLM prompt initialization

In [1]:
from chromadb.config import Settings
from chromadb import Client
from langchain.vectorstores import Chroma
import chromadb

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from typing_extensions import List, TypedDict
from langgraph.graph import START, StateGraph

import os, re
from datetime import datetime

date = datetime.today().strftime('%Y-%m-%d')

# Initialize Langsmith
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = "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"
os.environ["LANGSMITH_PROJECT"] = f"[{date}] VAA - Query Enhancement (RAG-Fusion)"

# Initialize LLM
REASONING = True

llm = ChatOllama(model="deepseek-r1:8b", validate_model_on_init=True, temperature=0.6, reasoning=True if REASONING else False)
emb = OllamaEmbeddings(model="bge-m3:567m")

In [2]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "polyu_eee_document" if not SINGLE else "vaa_documents"

# Initialize retriever for queries
client = Client(Settings())
client = chromadb.PersistentClient(path="../chroma_db")

vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=emb
)

client.list_collections()

/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_8802/3439006483.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(


[Collection(name=vaa_documents)]

In [3]:
# The LLM prompt
LLM_prompt = \
    """
    You are a professional academic advisor at The Hong Kong Polytechnic University. Please adhere to the following rules:
        1. Answer in the same language as the user query, e.g., English query, English answer.
        2. Avoid saying "may", "maybe", or anything similar; be affirmative, confident, and decisive in your answers.
        3. Avoid saying "based on the provided context", or anything similar; answer directly.
        4. Say no if you cannot answer the question; do not fabricate a factually false answer.
        5. Provide advice to the student if necessary.

    Now, please use the following context to answer the student's question.
    Remember to be nice and ask if there are any more enquiries.

    *Context*:
    ----------
    {context}
    ----------

    *Student's Question*:
    {question}

    Helpful Answer:
    """
prompt = PromptTemplate.from_template(LLM_prompt)

### RAG-Fusion Implementation

In [10]:
num_queries = 4 # Number of additional queries to generate in RAG-Fusion

# RAG-Fusion prompt
RAG_FUSION_PROMPT = \
    """
    You are a helpful assistant that generates multiple search questions based on a single input query.

    Provide {num_queries} search questions separated by newlines. Do not say anything else.

    *User Question*: 
    {question}
    
    {num_queries} alternative questions:
    """
query_gen_prompt = PromptTemplate.from_template(RAG_FUSION_PROMPT)

In [4]:
# Defining the class structure for the LLM
class State(TypedDict):
    question: str
    queries: List[str]
    context: List[Document]
    answer: str

# Functions for query generation
def generate_queries(state: State):
    question = state["question"]
    messages = query_gen_prompt.invoke({"question": question, "num_queries": num_queries})
    response = llm.invoke(messages)
    queries = [question] + response.content.strip().split("\n")
    return {"queries": queries}

# Functions for document retrieval based on cos-sim
def retrieve(state: State):
    all_docs = []
    
    for query in state["queries"]:
        retrieved_docs = vectorStore.similarity_search(query, k=3)
        all_docs.append(retrieved_docs)
    return {"context": all_docs}

# Function for fusing and reranking documents (source: RAG Tutorial)
def fuse_and_rerank(state: State, k: int = 60):
    fused_scores = {}
    
    for docs in state["context"]:
        for rank, doc in enumerate(docs):
            doc_str = doc.page_content
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            fused_scores[doc_str] += 1 / (rank + k)
            
    reranked_results = [
        (doc_str, score) for doc_str, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    
    unique_docs = {doc.page_content: doc for docs in state["context"] for doc in docs}.values()
    doc_map = {doc.page_content: doc for doc in unique_docs}
    
    reranked_docs = [doc_map[doc_str] for doc_str, _ in reranked_results]
    return {"context": reranked_docs}

# Functions for constructing the final LLM prompt
def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(messages)
    #print(response.additional_kwargs)

    # Include the reasoning part in the output
    return {"answer": f"<think>\n{response.additional_kwargs.get("reasoning_content", "")}</think>\n\n{response.content}"}

    # NOT Include the reasoning part
    #return {"answer": f"{response.content}"}

# Functions for graph building (a process sequence)
def graph_building():
    global graph
    graph_builder = StateGraph(State)
    
    graph_builder.add_node("generate_queries", generate_queries)
    graph_builder.add_node("retrieve", retrieve)
    graph_builder.add_node("fuse_and_rerank", fuse_and_rerank)
    graph_builder.add_node("generate", generate)

    graph_builder.add_edge(START, "generate_queries")
    graph_builder.add_edge("generate_queries", "retrieve")
    graph_builder.add_edge("retrieve", "fuse_and_rerank")
    graph_builder.add_edge("fuse_and_rerank", "generate")
    
    graph = graph_builder.compile()

graph_building()

### Testing

In [ ]:
query = \
"What is the potential career path for studing in BEng Scheme in EE?"

print(f"Generating {query}")
result = graph.invoke({"question": query})

print(f"\nSub-questions generated:\n{result['queries']}")
print(f"\nAnswer generated:\n{result['answer']}")

Generating What is the potential career path for studing in BEng Scheme in EE?

Answer generated:
<think>
Hmm, the user is asking about potential career paths for studying the BEng Scheme in Electrical Engineering at The Hong Kong Polytechnic University. 

First, I need to understand the context provided. The context mentions details about applying for a Secondary Major in Artificial Intelligence and Data Analytics, credit requirements, and selection mechanisms. While it doesn't directly address career paths, it does mention the BEng (Hons) in Electrical Engineering plus the Secondary Major in Artificial Intelligence and Data Analytics. 

Given that the user is interested in the BEng Scheme in EE, I can infer they're considering this program and want to know about its career prospects. The context also mentions a Cumulative GPA requirement of 2.70 for Secondary Major enrolment, which might be relevant if they plan to pursue that additional specialization.

The user seems to be a prospe